# Encoder ablation on the attack graph (Table 3)
Every variant, including the full method, is trained with the same seed, budget, schedule, behavior policy, and game step as `main.ipynb`, so each row differs from the full method in one factor:

| Variant | Change from the full method |
|---|---|
| `full` | none: Eq. (3), $\lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$, with the nested $z^S/z^\Theta$ split |
| `no_contrastive` | $\lambda_\Theta = 0$ |
| `binary_infonce` | binary InfoNCE in place of Soft-InfoNCE (positives: structured similarity above 0.8) |
| `raw_param` | Soft-InfoNCE with a raw-parameter distance kernel in place of $\tilde d_\Theta$ |
| `single_encoder` | no split: actor and critic see only the context embedding, widened by $d$ |
| `no_nextrep` | $\lambda_p = 0$ |
| `no_reward` | $\lambda_r = 0$ |

Each trained defender is then evaluated with `eval.ipynb`'s protocol against three attackers (the trained attacker, the myopic critic best response, and the critic's equilibrium column). The table reports the worst-case return $\widehat W$ with its standard error, and **oracle recovered**: $\widehat W$ as a fraction of the Oracle's worst-case return, where the Oracle is the full variant's critic equilibrium with privileged access to the state and context. The per-defender PPO best response is not part of this suite.

Checkpoints go to `ablation_ckpts/`. Each variant takes as long as one run of `main.ipynb`; to train variants in parallel, run copies of this notebook with different `TRAIN_VARIANTS`.

In [ ]:
%run Initialize.ipynb

## Settings (training must match `main.ipynb`, evaluation must match `eval.ipynb`)

In [ ]:
SEED = 20
GAMMA = 0.99
CRITIC_LR = 1e-5
PLAYER1_LR = 1e-4
PLAYER2_LR = 1e-5
ENCODER_LR = 5e-5
CRITIC_BATCH = 64
PLAYER1_BATCH = 128
PLAYER2_BATCH = 128
CONTEXT_BATCH = 256
TAU = 5e-4
SI_TEMPERATURE = 0.5
NUM_EPISODES = 5000
TRAIN_EPISODE_LENGTH = 50
UPDATE_EVERY = 5
UPDATE_START = 20
CKPT_DIR = "ablation_ckpts"

EVAL_EPISODES = 200
EVAL_EPISODE_LENGTH = 50
EVAL_CFG_SEED = 12345
EVAL_EP_SEED0 = 1000
ATTACKER_SUITE = ["Trained P2", "Critic-BR P2", "Equilibrium P2"]

## Variants

In [ ]:
# lr_, lp, lc: weights of L_r, L_p, and the contrastive term. contrastive: None | 'soft_struct' | 'binary' | 'soft_raw'.
# split: True = z^S (state filter) + z^Theta (encoder); False = context embedding only, widened by d.
VARIANTS = {
    "full":           dict(lr_=0.25, lp=0.25, lc=0.25, contrastive="soft_struct", split=True),
    "no_contrastive": dict(lr_=0.25, lp=0.25, lc=0.00, contrastive=None,          split=True),
    "binary_infonce": dict(lr_=0.25, lp=0.25, lc=0.25, contrastive="binary",      split=True),
    "raw_param":      dict(lr_=0.25, lp=0.25, lc=0.25, contrastive="soft_raw",    split=True),
    "single_encoder": dict(lr_=0.25, lp=0.25, lc=0.25, contrastive="soft_struct", split=False),
    "no_nextrep":     dict(lr_=0.25, lp=0.00, lc=0.25, contrastive="soft_struct", split=True),
    "no_reward":      dict(lr_=0.00, lp=0.25, lc=0.25, contrastive="soft_struct", split=True),
}
VARIANT_ORDER = list(VARIANTS.keys())


def emb_size_of(cfg):
    return Embedding_Size + (0 if cfg["split"] else d)


def ckpt_path(name, part):
    return os.path.join(CKPT_DIR, f"{name}_{part}.pth")


def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Variant networks (`full` uses the same networks and construction order as `main.ipynb`)

In [ ]:
class WideTransformer(TSTransformer):
    # Context encoder with a wider output, for the single-encoder variant
    def __init__(self, out_dim):
        super().__init__()
        self.fc2 = nn.Linear(128, out_dim)


class NoSplitCritic(nn.Module):
    # Critic without the z^S branch: the state filter is not an input
    def __init__(self, emb_size):
        super().__init__()
        self.enc_embd1 = nn.Linear(emb_size, 128); self.enc_embd2 = nn.Linear(128, 128)
        self.enc_state1 = nn.Linear(d, 128);       self.enc_state2 = nn.Linear(128, 128)
        self.enc_m2_1 = nn.Linear(d, 128);         self.enc_m2_2 = nn.Linear(128, 128)
        self.enc_type1 = nn.Linear(3, 64);         self.enc_type2 = nn.Linear(64, 128)
        self.enc_theta1 = nn.Linear(Len_Theta, 64); self.enc_theta2 = nn.Linear(64, 128)
        self.ln_fuse = nn.LayerNorm(128 * 5)
        self.fuse1 = nn.Linear(128 * 5, 512)
        self.fuse2 = nn.Linear(512, 128)
        self.out = nn.Linear(128, n_a * n_u)

    def forward(self, z_s, Embeding, x, m2, type_theta, theta):
        e = torch.cat([F.relu(self.enc_embd2(F.relu(self.enc_embd1(Embeding)))),
                       F.relu(self.enc_state2(F.relu(self.enc_state1(x)))),
                       F.relu(self.enc_m2_2(F.relu(self.enc_m2_1(m2)))),
                       F.relu(self.enc_type2(F.relu(self.enc_type1(type_theta)))),
                       F.relu(self.enc_theta2(F.relu(self.enc_theta1(theta))))], dim=-1)
        e = F.relu(self.fuse1(self.ln_fuse(e)))
        return self.out(F.relu(self.fuse2(e)))


class NoSplitDefender(nn.Module):
    # Defender on the context embedding only
    def __init__(self, emb_size):
        super().__init__()
        self.enc_emb1 = nn.Linear(emb_size, 128)
        self.enc_emb2 = nn.Linear(128, 128)
        self.fuse1 = nn.Linear(128, 256)
        self.ln = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head = nn.Linear(128, n_a)

    def forward(self, z_s, Embeding):
        h = F.relu(self.enc_emb2(F.relu(self.enc_emb1(Embeding))))
        h = self.ln(F.relu(self.fuse1(h)))
        return F.softmax(self.head(F.relu(self.fuse2(h))), dim=-1)


def make_models(cfg):
    # Same construction order as main.ipynb, so `full` starts from the same initialization
    emb = emb_size_of(cfg)
    M = {"cfg": cfg, "emb": emb}
    M["Context_Model"] = (TSTransformer() if cfg["split"] else WideTransformer(emb)).to(device)
    M["trans_pred"] = nn.Sequential(nn.Linear(emb + n_a + d, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                                    nn.Linear(256, emb)).to(device)
    M["reward_head"] = nn.Sequential(nn.Linear(emb, 128), nn.ReLU(), nn.Linear(128, 1)).to(device)
    M["proj_online"] = nn.Sequential(nn.Linear(emb, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
    M["pred_online"] = nn.Sequential(nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
    M["target_model"] = copy.deepcopy(M["Context_Model"])
    M["proj_target"] = copy.deepcopy(M["proj_online"])
    for p_ in list(M["target_model"].parameters()) + list(M["proj_target"].parameters()):
        p_.requires_grad_(False)
    M["target_model"].eval(); M["proj_target"].eval()
    M["encoder_optimizer"] = optim.AdamW(
        list(M["Context_Model"].parameters()) + list(M["trans_pred"].parameters()) +
        list(M["reward_head"].parameters()) + list(M["proj_online"].parameters()) +
        list(M["pred_online"].parameters()), lr=ENCODER_LR, amsgrad=True)

    make_critic = (lambda: Critic()) if cfg["split"] else (lambda: NoSplitCritic(emb))
    M["critic"] = make_critic().to(device)
    M["critic_target"] = make_critic().to(device)
    M["critic_target"].load_state_dict(M["critic"].state_dict())
    M["critic_optimizer"] = optim.AdamW(M["critic"].parameters(), lr=CRITIC_LR, amsgrad=True)

    M["Player1"] = (DefenderActor() if cfg["split"] else NoSplitDefender(emb)).to(device)
    M["player1_optimizer"] = optim.AdamW(M["Player1"].parameters(), lr=PLAYER1_LR, amsgrad=True)
    M["Player2"] = AttackerActor().to(device)
    M["player2_optimizer"] = optim.AdamW(M["Player2"].parameters(), lr=PLAYER2_LR, amsgrad=True)
    return M

## Contrastive objectives

In [ ]:
def raw_param_sim(th):
    # Raw-parameter ablation: Euclidean distance on theta, with a per-batch median bandwidth
    Dm = torch.cdist(th, th, p=2)
    eye = torch.eye(Dm.shape[0], device=th.device, dtype=torch.bool)
    sigma = Dm[~eye].median().clamp(min=1e-3).detach()
    return torch.exp(-Dm / sigma)


def soft_infonce(g, s, T=SI_TEMPERATURE):
    # L_SI = -sum_j w_ij log p_ij, the same form as Initialize.ipynb
    eye = torch.eye(g.shape[0], device=g.device, dtype=torch.bool)
    logits = ((g @ g.t()) / T).masked_fill(eye, -1e9)
    s = s.masked_fill(eye, 0.0)
    w = s / s.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return -(w * F.log_softmax(logits, dim=1)).sum(dim=1).mean()


def binary_infonce(g, s, thresh=0.8, T=SI_TEMPERATURE):
    # Standard InfoNCE with binary positives (similarity > thresh), averaged over anchors that have a positive
    eye = torch.eye(g.shape[0], device=g.device, dtype=torch.bool)
    pos = (s > thresh) & (~eye)
    valid = pos.any(dim=1)
    if valid.sum() == 0:
        return g.sum() * 0.0
    logits = ((g @ g.t()) / T).masked_fill(eye, -1e9)
    w = pos.float() / pos.float().sum(dim=1, keepdim=True).clamp(min=1.0)
    return (-(w * F.log_softmax(logits, dim=1)).sum(dim=1))[valid].mean()

## Training updates: `main.ipynb`'s representation and game steps, parameterized by the variant

In [ ]:
def ema_update_v(M, tau=0.995):
    for po, pt in zip(M["Context_Model"].parameters(), M["target_model"].parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for po, pt in zip(M["proj_online"].parameters(), M["proj_target"].parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for bo, bt in zip(M["Context_Model"].buffers(), M["target_model"].buffers()):
        bt.data.copy_(bo.data)
    for bo, bt in zip(M["proj_online"].buffers(), M["proj_target"].buffers()):
        bt.data.copy_(bo.data)


def train_encoder(M, context_memory):
    cfg = M["cfg"]
    if len(context_memory) < CONTEXT_BATCH:
        return 0.0
    (ctx, msk, nctx, nmsk, a, y, r, th) = map(list, zip(*context_memory.sample(CONTEXT_BATCH)))
    ctx, msk = torch.cat(ctx).to(device), torch.cat(msk).to(device)
    nctx, nmsk = torch.cat(nctx).to(device), torch.cat(nmsk).to(device)
    a = torch.stack(a).to(device)
    y, r, th = torch.stack(y).to(device).float(), torch.stack(r).to(device).float(), torch.stack(th).to(device).float()

    for k in ("Context_Model", "trans_pred", "reward_head", "proj_online", "pred_online"):
        M[k].train()
    z = M["Context_Model"](ctx, msk)
    loss = z.sum() * 0.0

    if cfg["lr_"] > 0:                                     # L_r
        loss = loss + cfg["lr_"] * F.mse_loss(M["reward_head"](z).squeeze(-1), r)
    if cfg["lp"] > 0:                                      # L_p
        z_hat = M["trans_pred"](torch.cat([z, F.one_hot(a, num_classes=n_a).float(), y], dim=-1))
        h_hat = F.normalize(M["pred_online"](M["proj_online"](z_hat)), p=2, dim=1)
        with torch.no_grad():
            h_tgt = F.normalize(M["proj_target"](M["target_model"](nctx, nmsk)), p=2, dim=1)
        loss = loss + cfg["lp"] * F.mse_loss(h_hat, h_tgt)
    if cfg["lc"] > 0 and cfg["contrastive"] is not None:   # contrastive term
        g = F.normalize(M["proj_online"](z), p=2, dim=1)
        if cfg["contrastive"] == "soft_struct":
            loss = loss + cfg["lc"] * soft_infonce(g, Context_Similarity(th))
        elif cfg["contrastive"] == "soft_raw":
            loss = loss + cfg["lc"] * soft_infonce(g, raw_param_sim(th))
        elif cfg["contrastive"] == "binary":
            loss = loss + cfg["lc"] * binary_infonce(g, Context_Similarity(th))

    M["encoder_optimizer"].zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(M["Context_Model"].parameters()) + list(M["trans_pred"].parameters()) +
        list(M["reward_head"].parameters()) + list(M["proj_online"].parameters()) +
        list(M["pred_online"].parameters()), max_norm=1.0)
    M["encoder_optimizer"].step()
    ema_update_v(M)
    M["Context_Model"].eval()
    return loss.item()


def _game_batch_v(M, exp_memory, batch_size):
    # Sample transitions and recompute the context embeddings with the current encoder
    (x, nx, zs, nzs, m2, nm2, a, u, r, tt, th, ctx, msk, nctx, nmsk) = map(list, zip(*exp_memory.sample(batch_size)))
    S = lambda v: torch.stack(v).to(device)
    C = lambda v: torch.cat(v, dim=0).to(device)
    M["Context_Model"].eval()
    with torch.no_grad():
        emb, nemb = M["Context_Model"](C(ctx), C(msk)), M["Context_Model"](C(nctx), C(nmsk))
    return dict(x=S(x), nx=S(nx), zs=S(zs), nzs=S(nzs), m2=S(m2), nm2=S(nm2), a=S(a), u=S(u),
                r=S(r), tt=S(tt), th=S(th), emb=emb, nemb=nemb)


def train_critic(M, exp_memory):
    if len(exp_memory) < CRITIC_BATCH:
        return 0.0
    b = _game_batch_v(M, exp_memory, CRITIC_BATCH)
    Bsz = b['x'].shape[0]
    q_values = M["critic"](b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th']).gather(
        1, (b['a'] * n_u + b['u']).unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        next_q = M["critic_target"](b['nzs'], b['nemb'], b['nx'], b['nm2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        V_next = torch.tensor([MatrixValue(next_q[i])[0] for i in range(Bsz)], dtype=torch.float32, device=device)
        target_values = b['r'] + GAMMA * V_next
    loss = F.smooth_l1_loss(q_values, target_values)
    M["critic_optimizer"].zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(M["critic"].parameters(), max_norm=1.0)
    M["critic_optimizer"].step()
    with torch.no_grad():
        for tgt, src in zip(M["critic_target"].parameters(), M["critic"].parameters()):
            tgt.data.mul_(1.0 - TAU).add_(TAU * src.data)
    return loss.item()


def train_player(M, exp_memory, which):
    # which = 1: defender toward the row strategy; which = 2: attacker toward the column strategy
    batch = PLAYER1_BATCH if which == 1 else PLAYER2_BATCH
    if len(exp_memory) < batch:
        return 0.0
    b = _game_batch_v(M, exp_memory, batch)
    Bsz = b['x'].shape[0]
    with torch.no_grad():
        Q = M["critic"](b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        tgt = torch.tensor(np.stack([MatrixValue(Q[i])[which] for i in range(Bsz)]), dtype=torch.float32, device=device)
    if which == 1:
        pred, net, opt = M["Player1"](b['zs'], b['emb']), M["Player1"], M["player1_optimizer"]
    else:
        pred, net, opt = M["Player2"](b['x'], b['m2'], b['tt']), M["Player2"], M["player2_optimizer"]
    loss = -(tgt * torch.log(pred + 1e-8)).sum(dim=1).mean()
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    opt.step()
    return loss.item()

## Training loop and checkpoints

In [ ]:
def train_variant(name):
    cfg = VARIANTS[name]
    os.makedirs(CKPT_DIR, exist_ok=True)
    set_all_seeds(SEED)
    M = make_models(cfg)
    exp_memory = ReplayMemory(5000)
    context_memory = ReplayMemory(500)
    a_eps, a_eps_decay, a_eps_end = 0.3, 0.99, 0.1
    All_Loss = np.zeros([4, NUM_EPISODES])
    T = lambda v, dtype=torch.float32: torch.tensor(v, dtype=dtype, device=device)

    pbar = tqdm(range(NUM_EPISODES), desc=f"train[{name}]")
    for i_episode in pbar:
        Theta, Type_Theta, D, rho = Sample_Context()
        x = np.random.randint(n)
        z_s = np.ones(d) / 2
        Context = np.zeros([Context_length, d * 3], dtype=np.float32)
        m2 = np.full(d, 0.2)
        counts = np.zeros(d)

        for t in range(TRAIN_EPISODE_LENGTH):
            mask = Context_Mask(t)
            Context_Tensor = T(Context).view(1, Context_length, d * 3)
            M["Context_Model"].eval()
            with torch.no_grad():
                Embeding = M["Context_Model"](Context_Tensor, mask).squeeze(0)
            z_s_Tensor, x_Tensor = T(z_s), T(State_Matrix[x, :])
            theta_Tensor, type_theta_Tensor, m2_Tensor = T(Theta), T(Type_Theta), T(m2)

            with torch.no_grad():
                Q = M["critic"](z_s_Tensor, Embeding, x_Tensor, m2_Tensor, type_theta_Tensor, theta_Tensor).cpu().numpy()
                _, a_dist, _ = MatrixValue(Q.reshape(n_a, n_u))
                a_dist = 0.8 * a_dist + 0.2 * np.full(n_a, 1 / n_a)
                u_dist = M["Player2"](x_Tensor, m2_Tensor, type_theta_Tensor).cpu().numpy()
            a_dist = a_dist / a_dist.sum()
            u_dist = u_dist / u_dist.sum()
            if np.random.rand() < a_eps:
                a = int(random.choices(range(n_a), weights=np.ones([n_a]) / n_a)[0])
            else:
                a = int(np.random.choice(n_a, p=a_dist))
            if np.random.rand() < Type_Theta[2]:
                u = int(random.choices(range(n_u), weights=np.ones(n_u) / n_u)[0])
            else:
                u = int(np.random.choice(n_u, p=u_dist))

            r = Reward(x, a, u, Theta, D, rho)
            next_x = Next_State(x, a, u, D, rho)
            next_m2, counts = m2_update(State_Matrix[x, :], State_Matrix[next_x, :], m2, counts)
            y = Observation(State_Matrix[next_x, :])
            next_z_s = Def_State_Blf(z_s, y, A[a])

            new_row = Context_Row(z_s, a, next_z_s)
            if t < Context_length:
                Context[t, :] = new_row
            else:
                Context[:-1, :] = Context[1:, :]
                Context[-1, :] = new_row

            if t > 0:
                next_mask = Context_Mask(t + 1)
                Next_Context_Tensor = T(Context).view(1, Context_length, d * 3)
                a_Tensor, u_Tensor, Reward_Tensor = T(a, torch.long), T(u, torch.long), T(r)
                exp_memory.push([x_Tensor.cpu(), T(State_Matrix[next_x, :]).cpu(), z_s_Tensor.cpu(), T(next_z_s).cpu(),
                                 m2_Tensor.cpu(), T(next_m2).cpu(), a_Tensor.cpu(), u_Tensor.cpu(), Reward_Tensor.cpu(),
                                 type_theta_Tensor.cpu(), theta_Tensor.cpu(),
                                 Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu()])
                context_memory.push([Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu(),
                                     a_Tensor.cpu(), T(y, torch.long).cpu(), Reward_Tensor.cpu(), theta_Tensor.cpu()])

            x, z_s, m2 = next_x, next_z_s, next_m2

            if i_episode >= UPDATE_START and t % UPDATE_EVERY == 0:
                All_Loss[0, i_episode] += train_encoder(M, context_memory)
                All_Loss[1, i_episode] += train_critic(M, exp_memory)
                All_Loss[2, i_episode] += train_player(M, exp_memory, 1)
                All_Loss[3, i_episode] += train_player(M, exp_memory, 2)

        a_eps = max(a_eps_end, a_eps * a_eps_decay)
        if (i_episode + 1) % 100 == 0:
            pbar.set_postfix(enc=f"{All_Loss[0, i_episode]:.2f}", cri=f"{All_Loss[1, i_episode]:.2f}",
                             p1=f"{All_Loss[2, i_episode]:.3f}", p2=f"{All_Loss[3, i_episode]:.3f}")
        if (i_episode + 1) % 500 == 0 or (i_episode + 1) == NUM_EPISODES:
            save_variant(name, M, All_Loss, i_episode + 1)


def save_variant(name, M, All_Loss, ep):
    torch.save(M["Context_Model"].state_dict(), ckpt_path(name, "encoder"))
    torch.save(M["critic"].state_dict(), ckpt_path(name, "critic"))
    torch.save(M["Player1"].state_dict(), ckpt_path(name, "player1"))
    torch.save(M["Player2"].state_dict(), ckpt_path(name, "player2"))
    np.save(os.path.join(CKPT_DIR, f"{name}_losses.npy"), All_Loss[:, :ep])


def load_variant(name):
    M = make_models(VARIANTS[name])
    for k, part in (("Context_Model", "encoder"), ("critic", "critic"), ("Player1", "player1"), ("Player2", "player2")):
        M[k].load_state_dict(torch.load(ckpt_path(name, part), map_location=device, weights_only=True))
        M[k].eval()
    return M


def variant_available(name):
    return all(os.path.exists(ckpt_path(name, part)) for part in ("encoder", "critic", "player1", "player2"))


def variant_done(name):
    # Trained only if the checkpoints exist and the loss log covers the full budget
    if not variant_available(name):
        return False
    lp = os.path.join(CKPT_DIR, f"{name}_losses.npy")
    try:
        return os.path.exists(lp) and np.load(lp).shape[1] >= NUM_EPISODES
    except Exception:
        return False

## Train (variants already trained to the full budget are skipped; set `FORCE = True` to retrain)

In [ ]:
TRAIN_VARIANTS = VARIANT_ORDER
FORCE = False

for name in TRAIN_VARIANTS:
    if not FORCE and variant_done(name):
        print(f"{name}: already trained to {NUM_EPISODES} episodes, skipping")
        continue
    print(f"training {name}: {VARIANTS[name]}")
    train_variant(name)

## Evaluation protocol

In [ ]:
def eval_episode_configs():
    cfg_rng = np.random.RandomState(EVAL_CFG_SEED)
    configs = []
    for _ in range(EVAL_EPISODES):
        p = cfg_rng.permutation(TERMINALS)
        s1_parent, s2_parent = int(p[0]), int(p[1])
        t0 = cfg_rng.randint(0, 2)
        t1 = cfg_rng.randint(1 - t0, 2)
        lam = cfg_rng.uniform(0.0, 0.5)
        x0 = cfg_rng.randint(n)
        configs.append((s1_parent, s2_parent, t0, t1, lam, x0))
    return configs


def run_eval_episodes(M, defender_kind, attacker_kind, configs):
    # defender_kind: 'Trained P1' (the variant's defender), 'Oracle' (its critic's equilibrium row), or 'Random P1'.
    # Defenders sample from their own distributions; attackers minimize the defender's payoff.
    ep_disc = []
    for i_episode, (s1_parent, s2_parent, t0, t1, lam, x0) in enumerate(configs):
        np.random.seed(EVAL_EP_SEED0 + i_episode); torch.manual_seed(EVAL_EP_SEED0 + i_episode)
        random.seed(EVAL_EP_SEED0 + i_episode)                       # Next_State draws from Python's random
        Theta, Type_Theta, D, rho = Context_From_Config(s1_parent, s2_parent, t0, t1, lam)
        th_T = torch.tensor(Theta, dtype=torch.float32, device=device)
        tt_T = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        x = x0
        z_s = np.ones(d) / 2
        Context = np.zeros([Context_length, d * 3], dtype=np.float32)
        m2 = np.full(d, 0.2)
        counts = np.zeros(d)
        disc_sum = 0.0
        for t in range(EVAL_EPISODE_LENGTH):
            ctx = torch.tensor(Context, dtype=torch.float32, device=device).view(1, Context_length, d * 3)
            with torch.no_grad():
                emb = M["Context_Model"](ctx, Context_Mask(t)).squeeze(0)
            z_s_T = torch.tensor(z_s, dtype=torch.float32, device=device)
            x_T = torch.tensor(State_Matrix[x, :], dtype=torch.float32, device=device)
            m2_T = torch.tensor(m2, dtype=torch.float32, device=device)
            with torch.no_grad():
                Q = M["critic"](z_s_T, emb, x_T, m2_T, tt_T, th_T).cpu().numpy().reshape(n_a, n_u)
                if defender_kind == "Trained P1":
                    a_dist = M["Player1"](z_s_T, emb).cpu().numpy().astype(np.float64)
                elif defender_kind == "Oracle":
                    a_dist = MatrixValue(Q)[1]
                elif defender_kind == "Random P1":
                    a_dist = np.full(n_a, 1.0 / n_a)
                else:
                    raise ValueError(defender_kind)
            a_dist = np.clip(a_dist, 0, None); a_dist = a_dist / a_dist.sum()

            if np.random.rand() < lam:
                u = int(np.random.choice(n_u))
            elif attacker_kind == "Trained P2":
                with torch.no_grad():
                    u_dist = M["Player2"](x_T, m2_T, tt_T).cpu().numpy()
                u = int(np.random.choice(n_u, p=u_dist / u_dist.sum()))
            elif attacker_kind == "Critic-BR P2":
                u = int(np.argmin(a_dist @ Q))
            elif attacker_kind == "Equilibrium P2":
                q = np.clip(MatrixValue(Q)[2], 0, None)
                u = int(np.random.choice(n_u, p=q / q.sum()))
            else:
                raise ValueError(attacker_kind)

            a = int(np.random.choice(n_a, p=a_dist))
            r = Reward(x, a, u, Theta, D, rho)
            next_x = Next_State(x, a, u, D, rho)
            disc_sum += (GAMMA ** t) * r
            next_m2, counts = m2_update(State_Matrix[x, :], State_Matrix[next_x, :], m2, counts)
            y = Observation(State_Matrix[next_x, :])
            next_z_s = Def_State_Blf(z_s, y, A[a])
            row = Context_Row(z_s, a, next_z_s)
            if t < Context_length:
                Context[t, :] = row
            else:
                Context[:-1, :] = Context[1:, :]
                Context[-1, :] = row
            x, z_s, m2 = next_x, next_z_s, next_m2
        ep_disc.append(disc_sum)
    return np.asarray(ep_disc)

## Evaluate (cached in `ablation_ckpts/eval_runs.npz`; set `RECOMPUTE = True` to rerun)

In [ ]:
PRETTY = {
    "full":           "Full method",
    "no_contrastive": r"No contrastive ($\lambda_\Theta = 0$)",
    "binary_infonce": "Binary InfoNCE",
    "raw_param":      r"Raw-parameter distance for $\tilde d_\Theta$",
    "single_encoder": r"Single encoder (no $z^S/z^\Theta$ split)",
    "no_nextrep":     r"$\lambda_p = 0$",
    "no_reward":      r"$\lambda_r = 0$",
    "Random":         "Uniform random",
}

variants = [v for v in VARIANT_ORDER if variant_available(v)]
missing = [v for v in VARIANT_ORDER if v not in variants]
if missing:
    print("MISSING (excluded):", missing)
assert "full" in variants, "train the full variant first"

RECOMPUTE = False
CACHE = os.path.join(CKPT_DIR, "eval_runs.npz")
configs = eval_episode_configs()

if os.path.exists(CACHE) and not RECOMPUTE:
    z = np.load(CACHE)
    runs = {}
    for key in z.files:
        v, atk = key.split("|")
        runs.setdefault(v, {})[atk] = z[key]
    print("loaded cache:", CACHE)
else:
    runs = {}
    for v in tqdm(variants, desc="variants"):
        M = load_variant(v)
        runs[v] = {atk: run_eval_episodes(M, "Trained P1", atk, configs)
                   for atk in tqdm(ATTACKER_SUITE, leave=False, desc=v)}
        del M
    Mref = load_variant("full")                          # Oracle and Random references, under the full variant's models
    runs["Oracle"] = {atk: run_eval_episodes(Mref, "Oracle", atk, configs) for atk in ATTACKER_SUITE}
    runs["Random"] = {atk: run_eval_episodes(Mref, "Random P1", atk, configs) for atk in ATTACKER_SUITE}
    del Mref
    np.savez(CACHE, **{f"{v}|{a}": arr for v, d_ in runs.items() for a, arr in d_.items()})
    print("saved cache:", CACHE)

## Table

In [ ]:
worst = {v: min(runs[v], key=lambda a: runs[v][a].mean()) for v in runs}   # attackers minimize the defender's return
W = {v: runs[v][worst[v]] for v in runs}
W_oracle = W["Oracle"].mean()

order = [v for v in VARIANT_ORDER if v in variants] + ["Random"]
hdr = f"{'Variant':<42}{'W_hat':>9}{'SE':>8}{'oracle recovered':>19}   worst attacker"
print(hdr); print("-" * len(hdr))
for v in order:
    plain = PRETTY[v].replace("$", "").replace("\\", "")
    se = W[v].std(ddof=1) / np.sqrt(len(W[v]))
    print(f"{plain:<42}{W[v].mean():>9.3f}{se:>8.3f}{W[v].mean() / W_oracle:>18.0%}   {worst[v]}")
print(f"\nOracle worst-case return: {W_oracle:.3f} (worst attacker {worst['Oracle']})")